In [ ]:
# Load env variables and create client
import os
from dotenv import load_dotenv
from anthropic import Anthropic
from statistics import mean

load_dotenv()

auth_token = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(
    auth_token=auth_token,
    base_url="https://flow.ciandt.com/flow-llm-proxy/"
)

model = "bedrock/anthropic.claude-4-5-haiku"

In [40]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [79]:
import json
import re


def extract_json_array(text):
    match = re.search(r'\[.*\]', text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON array found in response:\n{text}")
    return json.loads(match.group())


def extract_json_object(text):
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in response:\n{text}")
    return json.loads(match.group())


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects. Example format:
[
    {
        "task": "Description of task"
    }
]
"""
    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    return extract_json_array(text)


In [42]:
dataset = generate_dataset()

with open("database.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [59]:
def run_prompt(test_case):
    """"Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the follwing task

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [83]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer. Evaluate this AI-generated solution.

Task: {test_case["task"]}
Solution: {output}

Provide your evaluation as a structured JSON object with:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your assessment
- "score": A number between 1-10
"""

    messages = []
    add_user_message(messages, eval_prompt)
    eval_text = chat(messages)
    return extract_json_object(eval_text)


In [84]:
def run_test_case(test_case): 
    """"Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)

    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return { 
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [85]:
def run_eval(dataset):
    """"Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

In [86]:
with open("database.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results))

Average score: 6.833333333333333
[{"output": "# AWS Region Extraction from S3 URL\n\nHere's a comprehensive solution with multiple approaches:\n\n## Solution 1: Using Regular Expressions (Recommended)\n\n```python\nimport re\n\ndef extract_aws_region(s3_url):\n    \"\"\"\n    Extracts AWS region from an S3 bucket URL.\n    \n    Args:\n        s3_url (str): S3 URL in format like 's3://my-bucket.s3.us-west-2.amazonaws.com/key'\n    \n    Returns:\n        str: AWS region code (e.g., 'us-west-2'), or None if not found\n    \n    Examples:\n        >>> extract_aws_region('s3://my-bucket.s3.us-west-2.amazonaws.com/key')\n        'us-west-2'\n        >>> extract_aws_region('https://my-bucket.s3.eu-central-1.amazonaws.com/key')\n        'eu-central-1'\n    \"\"\"\n    pattern = r'\\.s3[.-]([a-z0-9-]+)\\.amazonaws\\.com'\n    match = re.search(pattern, s3_url)\n    return match.group(1) if match else None\n```\n\n## Solution 2: String Parsing (No Regex)\n\n```python\ndef extract_aws_region_si